# 02 — Data Cleaning and Preprocessing

**Goal:** clean the source tables, create reusable movie features, and save processed data for the recommendation models. Run Notebook 01 first.

## 1. Imports and paths

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Keep paths consistent whether the notebook is launched from the project root or 02_Notebooks.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "02_Notebooks" else CWD
DATA_RAW = PROJECT_ROOT / "01_Data" / "Raw"
DATA_PROCESSED = PROJECT_ROOT / "01_Data" / "Processed"
MODELS = PROJECT_ROOT / "03_Models"
VISUALS = PROJECT_ROOT / "05_Visualizations"
for folder in [DATA_RAW, DATA_PROCESSED, MODELS, VISUALS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

from pathlib import Path
RAW_DIR = DATA_RAW / "ml-latest-small"
movies = pd.read_csv(RAW_DIR / "movies.csv")
ratings = pd.read_csv(RAW_DIR / "ratings.csv")
tags = pd.read_csv(RAW_DIR / "tags.csv")
links = pd.read_csv(RAW_DIR / "links.csv")
print(movies.shape, ratings.shape, tags.shape, links.shape)

Project root: D:\DataScience\notebooks\CineMatch_Ready_Notebooks
(9742, 3) (100836, 4) (3683, 4) (9742, 3)


## 2. Remove duplicates and handle missing values

MovieLens normally has very few missing values. We remove exact duplicate rows, drop records missing essential identifiers/ratings, and retain only valid rating values from 0.5 to 5.0.

In [2]:
movies = movies.drop_duplicates(subset="movieId").copy()
ratings = ratings.drop_duplicates(subset=["userId", "movieId"]).copy()
tags = tags.drop_duplicates().copy()
links = links.drop_duplicates(subset="movieId").copy()

movies = movies.dropna(subset=["movieId", "title", "genres"])
ratings = ratings.dropna(subset=["userId", "movieId", "rating"])
ratings = ratings[ratings["rating"].between(0.5, 5.0)].copy()
tags = tags.dropna(subset=["movieId", "tag"]).copy()

for frame in [movies, ratings, tags, links]:
    frame["movieId"] = frame["movieId"].astype("int64")
ratings["userId"] = ratings["userId"].astype("int64")
tags["userId"] = tags["userId"].astype("int64")
print("Clean shapes:", movies.shape, ratings.shape, tags.shape, links.shape)

Clean shapes: (9742, 3) (100836, 4) (3683, 4) (9742, 3)


## 3. Clean titles and extract release year

Movie titles often end with a year in parentheses. We extract it where present and preserve the original title for display.

In [3]:
movies["title"] = movies["title"].astype(str).str.strip()
movies["release_year"] = pd.to_numeric(
    movies["title"].str.extract(r"\\((\\d{4})\\)\\s*$")[0], errors="coerce"
).astype("Int64")
movies["clean_title"] = movies["title"].str.replace(r"\\s*\\(\\d{4}\\)\\s*$", "", regex=True).str.strip()
movies["genres"] = movies["genres"].fillna("(no genres listed)").astype(str).str.strip()
display(movies[["movieId", "title", "clean_title", "release_year", "genres"]].head())

,movieId,title,clean_title,release_year,genres
0,1,Toy Story (1995),Toy Story (1995),<NA>,Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Jumanji (1995),<NA>,Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Grumpier Old Men (1995),<NA>,Comedy|Romance
3,4,Waiting to Exhale (1995),Waiting to Exhale (1995),<NA>,Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Father of the Bride Part II (1995),<NA>,Comedy


## 4. Process genres and tags

Genres are pipe-separated labels. Tags are combined per movie into a single text field. We do not use tags as ratings; they are descriptive metadata.

In [4]:
movies["genre_list"] = movies["genres"].apply(
    lambda x: [g for g in x.split("|") if g and g != "(no genres listed)"]
)
tags["tag_clean"] = (tags["tag"].astype(str).str.lower()
                     .str.replace(r"[^a-z0-9\\s]", " ", regex=True)
                     .str.replace(r"\\s+", " ", regex=True).str.strip())
movie_tags = (tags[tags["tag_clean"].ne("")]
              .groupby("movieId")["tag_clean"]
              .apply(lambda values: " ".join(values))
              .rename("tags_text").reset_index())
movies = movies.merge(movie_tags, on="movieId", how="left")
movies["tags_text"] = movies["tags_text"].fillna("")
movies["content_text"] = (
    movies["clean_title"].fillna("") + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["tags_text"]
).str.lower().str.replace(r"\\s+", " ", regex=True).str.strip()
display(movies[["title", "genres", "tags_text", "content_text"]].head())

,title,genres,tags_text,content_text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,toy story (1995) adventure animation children ...
1,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game robin williams game,jumanji (1995) adventure children fantasy fant...
2,Grumpier Old Men (1995),Comedy|Romance,moldy old,grumpier old men (1995) comedy romance moldy old
3,Waiting to Exhale (1995),Comedy|Drama|Romance,,waiting to exhale (1995) comedy drama romance
4,Father of the Bride Part II (1995),Comedy,pregnancy remake,father of the bride part ii (1995) comedy preg...


## 5. Merge external links and calculate rating summaries

The movie catalogue is the left-hand table so movies with no ratings remain available for content-based recommendations.

In [5]:
movies = movies.merge(links, on="movieId", how="left")
rating_summary = ratings.groupby("movieId").agg(
    average_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()
movies = movies.merge(rating_summary, on="movieId", how="left")
movies["average_rating"] = movies["average_rating"].fillna(0.0)
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)
print("Movies with no ratings:", int((movies["rating_count"] == 0).sum()))

Movies with no ratings: 18


## 6. Prepare model tables and save processed datasets

In [6]:
# Keep a clean user-movie interaction table; missing ratings are not converted to zero.
user_movie_ratings = ratings[["userId", "movieId", "rating", "timestamp"]].copy()
content_movies = movies[[
    "movieId", "title", "clean_title", "release_year", "genres",
    "genre_list", "tags_text", "content_text", "average_rating",
    "rating_count", "imdbId", "tmdbId"
]].copy()

content_movies.to_pickle(DATA_PROCESSED / "movies_processed.pkl")
content_movies.to_csv(DATA_PROCESSED / "movies_processed.csv", index=False)
user_movie_ratings.to_csv(DATA_PROCESSED / "ratings_processed.csv", index=False)
tags.to_csv(DATA_PROCESSED / "tags_processed.csv", index=False)
print("Saved processed movies and ratings to:", DATA_PROCESSED)

Saved processed movies and ratings to: D:\DataScience\notebooks\CineMatch_Ready_Notebooks\01_Data\Processed


## 7. Final validation

Confirm the IDs are unique in the movie table, ratings are valid, and all saved model tables have the expected columns.

In [7]:
assert content_movies["movieId"].is_unique
assert user_movie_ratings["rating"].between(0.5, 5.0).all()
assert user_movie_ratings[["userId", "movieId", "rating"]].notna().all().all()
print("Validation passed.")
print("Movies:", content_movies.shape, "| Ratings:", user_movie_ratings.shape)
display(content_movies.head(3))

Validation passed.
Movies: (9742, 12) | Ratings: (100836, 4)


,movieId,title,clean_title,release_year,genres,genre_list,tags_text,content_text,average_rating,rating_count,imdbId,tmdbId
0,1,Toy Story (1995),Toy Story (1995),<NA>,Adventure|Animation|Children|Comedy|Fantasy,"[Adventure, Animation, Children, Comedy, Fantasy]",pixar pixar fun,toy story (1995) adventure animation children ...,3.920930,215,114709,862.0
1,2,Jumanji (1995),Jumanji (1995),<NA>,Adventure|Children|Fantasy,"[Adventure, Children, Fantasy]",fantasy magic board game robin williams game,jumanji (1995) adventure children fantasy fant...,3.431818,110,113497,8844.0
2,3,Grumpier Old Men (1995),Grumpier Old Men (1995),<NA>,Comedy|Romance,"[Comedy, Romance]",moldy old,grumpier old men (1995) comedy romance moldy old,3.259615,52,113228,15602.0


In [8]:
# Install Plotly in the current Jupyter environment.

%pip install plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -----------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Completion check

Required outputs: `01_Data/Processed/movies_processed.pkl`, `movies_processed.csv`, and `ratings_processed.csv`. Next: **03_Exploratory_Data_Analysis.ipynb**.